# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


In [2]:
from nba_ou.modeling.modeling import (
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
)

In [3]:
nan_threshold = 5
max_na_per_row = 3

## Load Data

In [4]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260318.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [5]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11258 rows
Basic cleaning complete: 8649 rows remaining

Starting advanced column cleaning with 3240 columns

Advanced column cleaning complete: 3240 → 1358 columns (1882 removed)


Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 104
  Infer pairs applied: 20/136
  Remaining NaN cells: 127364

Dropping rows with more than 3 NaN values...
Removed 1395 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (7254, 1358)


In [6]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1155,LEAGUE_GAMES_LAST_1D_BEFORE,266,3.67,2023.0
1348,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D_BEFORE,79,1.09,2019.0
1345,PTS_FORM_Z_HOME_LAST5_VS_SEASON_BEFORE,63,0.87,2022.0
216,ml_betmgm_price_LAST_ALL_5_MATCHES_BEFORE_TEAM...,62,0.85,2019.0
628,ml_betmgm_price_LAST_ALL_5_MATCHES_BEFORE_TEAM...,60,0.83,2019.0
236,PTS_TREND_SLOPE_LAST_5_HOME_AWAY_GAMES_BEFORE_...,59,0.81,2020.0
1346,PTS_FORM_Z_AWAY_LAST5_VS_SEASON_BEFORE,56,0.77,2020.0
648,PTS_TREND_SLOPE_LAST_5_HOME_AWAY_GAMES_BEFORE_...,54,0.74,2020.0
1347,TRAVEL_RECENCY_RATIO_HOME_2D_OVER_14D_BEFORE,42,0.58,2019.0
883,DIFF_FROM_LINE_caesars_LAST_ALL_1_MATCHES_DIFF...,28,0.39,2023.0


In [7]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [8]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [9]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2019     296
2020    1098
2021    1231
2022    1224
2023    1186
2024    1250
2025     969
dtype: int64


## Train / Test

In [10]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.base import clone

In [11]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.025,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 7061
Final test set size: 184
Final test date range: 2026-02-21 00:00:00 -> 2026-03-17 00:00:00


In [12]:
TARGET_COL = "TOTAL_POINTS"

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

X_dev shape: (7061, 1355)
X_test_final shape: (184, 1355)


In [13]:
from nba_ou.modeling.scorers import OverUnderScorerTotalPoints, over_under_betting_accuracy_total_points, evaluate_total_points_thresholds
    
ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
}

def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc == "OU_Betting_Accuracy":
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()

In [14]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=30,
    step_games_between_tests=50,
    train_games=7500,
    min_train_games=6000,
    max_folds=12,
    verbose=1,
)


Created 12 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           6003            30       2019-11-10     2025-03-17      2025-03-18    2025-03-21         2024
    2           6083            30       2019-11-10     2025-03-28      2025-03-29    2025-04-01         2024
    3           6165            44       2019-11-10     2025-04-08      2025-04-09    2025-04-13         2024
    4           6285            34       2019-11-10     2025-06-22      2025-10-26    2025-11-01         2025
    5           6373            34       2019-11-10     2025-11-08      2025-11-09    2025-11-12         2025
    6           6459            34       2019-11-10     2025-11-20      2025-11-21    2025-11-24         2025
    7           6547            33       2019-11-10     2025-12-02      2025-12-03    2025-12-06         2025
    8           6634            42       2019-11-10     2025-12-18      2025

In [15]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MSE: 379.40070
Validation MSE: 402.02801

Train RMSE: 19.47820
Validation RMSE: 19.87511

Train MAE: 15.52100
Validation MAE: 15.98349

Train R2: 0.00000
Validation R2: -0.08551

Train OU_Betting_Accuracy: 50.27%
Validation OU_Betting_Accuracy: 48.84%



In [16]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MSE: 235.39489
Validation MSE: 9147249.94875

Train RMSE: 15.34225
Validation RMSE: 891.53695

Train MAE: 12.15199
Validation MAE: 161.14861

Train R2: 0.37956
Validation R2: -14605.02967

Train OU_Betting_Accuracy: 64.28%
Validation OU_Betting_Accuracy: 51.85%



In [17]:
xgb_reg = XGBRegressor(
    max_depth=4,
    learning_rate=0.047,
    n_estimators=115,
    subsample=0.8,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-2,          # choose your CPU count
    random_state=16,
)

cv_results = cross_validate(
    xgb_reg,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,          # leave at 1 because XGB already parallelizes internally
)

print("XGBoost")
print_metrics(cv_results)

XGBoost
Train MSE: 216.48780
Validation MSE: 310.48155

Train RMSE: 14.71301
Validation RMSE: 17.41145

Train MAE: 11.64161
Validation MAE: 13.93128

Train R2: 0.42940
Validation R2: 0.15303

Train OU_Betting_Accuracy: 75.38%
Validation OU_Betting_Accuracy: 54.18%



In [18]:
xgb_reg.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_test_final,
    y_pred_test_total,
    betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 319.41525
RMSE: 17.87219
MAE: 13.91219
OU_Betting_Accuracy: 52.22%


In [19]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,184,100.0%,52.22%
1,1,137,74.5%,54.48%
2,2,92,50.0%,53.33%
3,3,51,27.7%,53.06%
4,4,10,5.4%,50.00%
5,5,7,3.8%,28.57%
6,6,2,1.1%,0.00%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


# OPTUNA

In [20]:
from nba_ou.modeling.optuna_total_points import (
    tune_xgb_total_points_optuna,
    summarize_optuna_trials,
    fit_best_xgb_total_points,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=4 * 3600,
    objective_name="reg:squarederror",   # second run: reg:pseudohubererror
    study_name="xgb_total_points_mae",
)

print("Best CV MAE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\nSecondary metrics from best trial:")
print("Mean RMSE:", study.best_trial.user_attrs.get("mean_rmse"))
print("Mean R2:", study.best_trial.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean best_iteration:", study.best_trial.user_attrs.get("mean_best_iteration"))

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
        }
    )
)

[I 2026-03-18 17:46:18,134] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-18 17:52:23,294] Trial 0 finished with value: 13.710816079373087 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307}. Best is trial 0 with value: 13.710816079373087.
[I 2026-03-18 18:01:31,196] Trial 1 finished with value: 13.66076298724065 and parameters: {'max_depth': 2, 'min_child_weight': 51.819268381786635, 'gamma': 1.7346760026809933, 'subsample': 0.5811969357559948, 'colsample_bytree': 0.675188230006178, 'learning_rate': 0.010426963054371881, 'reg_alpha': 0.06701717253228541, 'reg_lambda': 3.152289132591451}. Best is trial 1 with value: 13.66076298724065.
[I 2026-03-18 18:08:38,454] Trial 2 finished with value: 13.454210984725497 and parameters: {'max_depth': 4, 'min_child_weight': 15.848753157115912, 'gamma': 0.7236802167715846, 'subsample': 0

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda
0,24,13.3942,17.1718,0.1845,56.80%,99,4,33.227428,1.232740,0.767378,0.557692,0.049429,0.439351,6.079059
1,11,13.4008,17.1938,0.1836,59.44%,115,4,22.732528,0.685930,0.899653,0.753025,0.058232,0.284370,4.794586
2,21,13.4140,17.1416,0.1873,57.67%,108,4,15.163190,0.766109,0.730259,0.435382,0.050181,0.704200,7.187449
3,13,13.4337,17.2176,0.1798,59.62%,104,4,12.295994,0.645851,0.678114,0.799424,0.047061,0.188925,7.345288
4,33,13.4451,17.2115,0.1817,59.00%,102,4,13.679111,0.705420,0.716287,0.680291,0.049037,0.150942,3.633642
5,2,13.4542,17.2258,0.1799,57.48%,88,4,15.848753,0.723680,0.728731,0.404395,0.050561,0.741065,6.337919
6,23,13.4570,17.1756,0.1840,58.22%,121,4,23.660749,0.893860,0.723966,0.454251,0.031779,0.106258,4.006453
7,31,13.4576,17.1920,0.1827,56.21%,84,4,16.254311,0.607580,0.698289,0.758476,0.051363,0.268598,5.882731
8,8,13.4582,17.1963,0.1832,57.73%,119,4,18.522692,1.744264,0.839178,0.657172,0.039201,0.023578,2.179573
9,17,13.4589,17.1445,0.1890,59.06%,85,4,6.030260,1.314574,0.651825,0.741301,0.045905,0.489133,4.058369


In [21]:
best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    study=study,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 316.09769
RMSE: 17.77914
MAE: 13.75666
OU_Betting_Accuracy: 52.78%


In [22]:
from nba_ou.modeling.modeling import (
    ModelBundleMetadata,
    ModelInfo,
    TrainingMetrics,
    save_model_bundle,
)

X_full = df_to_train.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train[TARGET_COL], errors="coerce")

production_model = fit_best_xgb_total_points(
    X_dev=X_full,
    y_dev=y_full,
    study=study,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"full_xgb_total_points_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="full_dataset_total_points",
        prediction_source="full_xgb_total_points",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=study.best_trial.params,
        mean_best_iteration=study.best_trial.user_attrs.get("mean_best_iteration"),
        cv_mae=float(study.best_value),
        cv_rmse=study.best_trial.user_attrs.get("mean_rmse"),
        cv_ou_acc=study.best_trial.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train["GAME_DATE"].max().to_pydatetime(),
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/full_dataset/",
    metadata=metadata,
)

print(f"Production model trained on {len(X_full)} rows using fixed n_estimators from mean_best_iteration.")
print("Saved model :", model_path)
print("Saved metadata:", meta_path)


Production model trained on 7245 rows using fixed n_estimators from mean_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/full_dataset/full_xgb_total_points_17_03_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/full_dataset/full_xgb_total_points_17_03_26.meta.json
